# Домашнє завдання: Рекомендаційні системи на реальних даних (Goodbooks-10k)

У цьому завданні Ви реалізуєте сучасні (advanced) архітектури рекомендаційних систем із фінального блоку лекції — але вже **не на іграшкових даних, а на реальному датасеті книжкових рейтингів Goodbooks-10k** (десятки тисяч користувачів, тисячі книг, мільйони оцінок).

Це дасть Вам змогу побачити, як підходи поводяться, коли даних справді багато: чому контентних ознак буває замало, як працює retrieval на тисячах елементів, і чому офлайн-метрики на кшталт Recall@K не такі високі, як хотілося б.

**Архітектури, які Ви зберете:** Vector Space Model, Two-Tower, Concat-based ranking (NCF) та двоетапний пайплайн Retrieval → Ranking.

**Стек:** `numpy`, `pandas`, `scikit-learn`, `torch`. GPU не обов'язковий, але з ним тренування буде швидшим (у Colab: *Runtime → Change runtime type → GPU*).

---

## Про датасет

[Goodbooks-10k](https://www.kaggle.com/datasets/zygmunt/goodbooks-10k) — це ~6 млн оцінок 10 000 найпопулярніших книг від 53 424 користувачів. Складається з кількох файлів:

- `ratings.csv` — оцінки: `user_id, book_id, rating` (1–5);
- `books.csv` — метадані книг: `book_id, goodreads_book_id, authors, title, average_rating, ...`;
- `book_tags.csv` — теги/полиці, які користувачі вішали на книги: `goodreads_book_id, tag_id, count`;
- `tags.csv` — розшифровка тегів: `tag_id, tag_name`.

**Важливий нюанс:** на відміну від навчального прикладу, тут **немає готових жанрів**. Жанри доведеться сконструювати самостійно з користувацьких тегів — а це шумні дані (юзери можуть зазначати що завгодно). Це реалістична задача feature engineering, і ми її розберемо в підготовчій частині.

Ще один нюанс із реальних даних: `book_tags.csv` посилається на `goodreads_book_id`, а `ratings.csv` — на `book_id`. Щоб їх поєднати, потрібен джойн через `books.csv`.


## Крок 0. Завантаження даних

Є три способи дістати дані — оберіть будь-який.

**Спосіб A — Kaggle API (рекомендований).** Завантаження з Kaggle API. Зручно, бо декілька файлів і вони завантажаться всі самостійно. Для цього способу завантажте свій `kaggle.json` (Kaggle → Account → Create New API Token), потім виконайте:
```python
from google.colab import files; files.upload()   # оберіть kaggle.json
```
і розкоментуйте відповідний блок нижче.

**Спосіб B — ручне завантаження.** Завантажте архів з посилання на датасет вище з Kaggle, розпакуйте і покладіть `ratings.csv`, `books.csv`, `book_tags.csv`, `tags.csv` поруч із ноутбуком (або через панель Files у Colab).

**Спосіб C — GitHub-дзеркало (фолбек).** Оригінальний автор виклав файли і на GitHub — код нижче підхопить їх автоматично, якщо локально файлів немає.


In [ ]:
# (Спосіб A) Kaggle API — розкоментуйте, якщо завантажили kaggle.json
# !pip -q install kaggle
# import os, shutil
# os.makedirs("/root/.kaggle", exist_ok=True)
# shutil.move("kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
# !kaggle datasets download -d zygmunt/goodbooks-10k --unzip -p .

In [1]:
import os
import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master"
FILES = ["ratings.csv", "books.csv", "book_tags.csv", "tags.csv"]

def load(fname):
    """Спочатку шукаємо файл локально, інакше тягнемо з GitHub-дзеркала."""
    if os.path.exists(fname):
        return pd.read_csv(fname)
    print(f"{fname} не знайдено локально — завантажую з GitHub...")
    return pd.read_csv(f"{GITHUB}/{fname}")

ratings = load("ratings.csv")
books = load("books.csv")
book_tags = load("book_tags.csv")
tags = load("tags.csv")

print("ratings:", ratings.shape)
print("books:  ", books.shape)
print("book_tags:", book_tags.shape, "| tags:", tags.shape)
books[["book_id", "authors", "title", "average_rating"]].head()

ratings.csv не знайдено локально — завантажую з GitHub...
books.csv не знайдено локально — завантажую з GitHub...
book_tags.csv не знайдено локально — завантажую з GitHub...
tags.csv не знайдено локально — завантажую з GitHub...
ratings: (5976479, 3)
books:   (10000, 23)
book_tags: (999912, 3) | tags: (34252, 2)


,book_id,authors,title,average_rating
0,1,Suzanne Collins,"The Hunger Games (The Hunger Games, #1)",4.34
1,2,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,4.44
2,3,Stephenie Meyer,"Twilight (Twilight, #1)",3.57
3,4,Harper Lee,To Kill a Mockingbird,4.25
4,5,F. Scott Fitzgerald,The Great Gatsby,3.89


## Крок 1. Інженерія жанрів із тегів (feature engineering)

Жанрів у датасеті немає, але є користувацькі теги. Виберемо набір канонічних жанрів і для кожної книги позначимо, які з них їй приписали користувачі. Так ми отримаємо **бінарну матрицю book × genre** — це й будуть контентні ознаки айтемів (аналог `movie_feats_df` із лекції, але здобутий з реальних шумних даних).


In [2]:
# Канонічні жанри, які шукаємо серед тегів
GENRES = ["fantasy", "romance", "mystery", "thriller", "horror", "historical",
          "science-fiction", "young-adult", "nonfiction", "classics",
          "contemporary", "crime"]

# tag_name -> tag_id
name_to_tagid = dict(zip(tags["tag_name"], tags["tag_id"]))
genre_tag_ids = {g: name_to_tagid[g] for g in GENRES if g in name_to_tagid}

# book_tags використовує goodreads_book_id -> мапимо у book_id через books.csv
gid_to_bid = dict(zip(books["goodreads_book_id"], books["book_id"]))
tagid_to_genre = {tid: g for g, tid in genre_tag_ids.items()}

bt = book_tags[book_tags["tag_id"].isin(genre_tag_ids.values())].copy()
bt["book_id"] = bt["goodreads_book_id"].map(gid_to_bid)
bt = bt.dropna(subset=["book_id"])
bt["genre"] = bt["tag_id"].map(tagid_to_genre)

# бінарна матриця book × genre (жанр присутній, якщо користувачі його тегали)
genre_matrix = (
    bt.pivot_table(index="book_id", columns="genre", values="count", aggfunc="sum", fill_value=0)
      .reindex(columns=GENRES, fill_value=0) > 0
).astype(int)

print("Книг із хоча б одним жанром:", (genre_matrix.sum(axis=1) > 0).sum(), "/", len(books))
print("\nРозподіл жанрів:")
print(genre_matrix.sum().sort_values(ascending=False))
genre_matrix.head()

Книг із хоча б одним жанром: 9954 / 10000

Розподіл жанрів:
genre
contemporary       5287
fantasy            4259
romance            4251
mystery            3686
young-adult        3630
classics           2785
historical         2544
thriller           2522
science-fiction    2222
crime              2083
nonfiction         1833
horror             1372
dtype: int64


genre,fantasy,romance,mystery,thriller,horror,historical,science-fiction,young-adult,nonfiction,classics,contemporary,crime
book_id,,,,,,,,,,,,
1,1,1,0,1,0,0,1,1,0,0,1,0
2,1,0,1,0,0,0,0,1,0,1,1,0
3,1,0,0,0,1,0,1,1,0,0,1,0
4,0,0,1,0,0,1,0,1,0,1,1,1
5,0,1,0,0,0,1,0,1,0,1,0,0


## Крок 2. Підвибірка під Colab

6 млн рейтингів — забагато для навчального ноутбука на CPU. Візьмемо **топ-N найпопулярніших книг** і **активних користувачів** (хто поставив ≥ 20 оцінок), а тоді обмежимо число користувачів. Так зберігається щільність взаємодій, а тренування лишається швидким.

> Якщо у Вас GPU або багато часу — сміливо збільшуйте `TOP_BOOKS` та `N_USERS`.


In [3]:
TOP_BOOKS = 1500       # скільки найпопулярніших книг лишити
MIN_USER_RATINGS = 20  # мінімум оцінок на користувача
N_USERS = 2000         # скільки користувачів узяти у підвибірку
LIKE_THRESHOLD = 4     # rating >= 4 вважаємо "лайком" (позитивна взаємодія)

rng = np.random.RandomState(42)

top_books = ratings["book_id"].value_counts().head(TOP_BOOKS).index
r = ratings[ratings["book_id"].isin(top_books)]
active = r["user_id"].value_counts()
r = r[r["user_id"].isin(active[active >= MIN_USER_RATINGS].index)]
sample_users = rng.choice(r["user_id"].unique(), size=min(N_USERS, r["user_id"].nunique()), replace=False)
r = r[r["user_id"].isin(sample_users)].copy()

# лишаємо тільки книги, для яких є жанрові ознаки
r = r[r["book_id"].isin(genre_matrix.index)].copy()

items = sorted(r["book_id"].unique())
users = sorted(r["user_id"].unique())
genre_matrix = genre_matrix.reindex(items).fillna(0).astype(int)

print(f"Взаємодій: {len(r):,} | користувачів: {len(users):,} | книг: {len(items):,}")
print(f"Щільність: {len(r) / (len(users) * len(items)):.4f}")

Взаємодій: 140,934 | користувачів: 2,000 | книг: 1,496
Щільність: 0.0471


In [4]:
import torch
import torch.nn as nn

torch.manual_seed(42)

user_to_idx = {u: i for i, u in enumerate(users)}
item_to_idx = {b: i for i, b in enumerate(items)}
title_of = dict(zip(books["book_id"], books["title"]))

item_feats = torch.tensor(genre_matrix.values, dtype=torch.float32)  # (M, n_genres)
M = len(items)
n_genres = item_feats.shape[1]

# train/val split по взаємодіях
r = r.sample(frac=1, random_state=42).reset_index(drop=True)
n_val = int(len(r) * 0.2)
val_df = r.iloc[:n_val]
train_df = r.iloc[n_val:]

# позитивні пари (лайки) у train
train_pos = train_df[train_df["rating"] >= LIKE_THRESHOLD]
pos_u = torch.tensor([user_to_idx[u] for u in train_pos["user_id"]])
pos_i = torch.tensor([item_to_idx[b] for b in train_pos["book_id"]])

# що користувач уже бачив (щоб не рекомендувати повторно і не семплити як негатив)
from collections import defaultdict
seen_by_user = defaultdict(set)
for u, b in zip(train_df["user_id"], train_df["book_id"]):
    seen_by_user[user_to_idx[u]].add(item_to_idx[b])

# val-лайки для оцінки якості
val_pos = defaultdict(set)
for row in val_df.itertuples():
    if row.rating >= LIKE_THRESHOLD:
        val_pos[user_to_idx[row.user_id]].add(item_to_idx[row.book_id])

print(f"Позитивних пар у train: {len(pos_u):,} | користувачів з val-лайками: {len(val_pos):,}")

Позитивних пар у train: 77,070 | користувачів з val-лайками: 1,991


## Крок 3. Метрика оцінки якості рангування

В лекції ми з вами для оцінки якості використовували **RMSE**. Це валідний варіант, коли треба швидко оцінити якість рек. моделі, але спрощений. RMSE показує, наскільки точно модель передбачає оцінку, яку користувач поставить елементу.

В реальних системах нас ще цікавить **якість ранжування** — наскільки релевантні елементи потрапили в топ списку, який ми реально показуємо користувачу. Для цього використовують ранжувальні метрики: **Precision@K**, **Recall@K**, **NDCG**, **MAP**, **MRR**.

Детальніше можна познайомитись з цими мериками тут:
- огляд метрик для рекомендаційних систем: https://www.evidentlyai.com/ranking-metrics/evaluating-recommender-systems
- Precision та Recall at K: https://www.evidentlyai.com/ranking-metrics/precision-recall-at-k

Нижче давайте реалізуємо функцію `recall_at_k` і будемо оцінювати нею всі наші моделі.

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b2e_6577812c4d677925f1ab5f84_precision_recall_k9.png)

![](https://cdn.prod.website-files.com/660ef16a9e0687d9cc27474a/662c4327f27ee08d3e4d4b47_657781b1f9c868e0cda088f6_precision_recall_k11.png)

**Як працює `recall_at_k`:**

1. Для кожного користувача ми беремо його реальні вподобання з валідаційної вибірки (`val_pos` — книги, які він оцінив на ≥ 4), просимо модель оцінити всі книги й відбираємо топ-K рекомендацій. Перед цим прибираємо книги, які користувач уже бачив у train (щоб не рекомендувати відоме).

2. Далі рахуємо, скільки книг із топ-K справді потрапили в його вподобання (`hits`), і ділимо на загальну кількість релевантних книг (обмежену K, бо більше за K у топ і не влізе).

3. Усереднюємо по всіх користувачах — і отримуємо одне число від 0 до 1: **яку частку того, що користувачу реально сподобалось, модель змогла підняти в топ-K.**

In [5]:
def recall_at_k(score_fn, k=10):
    """Частка val-лайків, що потрапили у топ-k рекомендацій (усереднена по користувачах).
    score_fn(user_idx_tensor) -> матриця оцінок (n_users, M)."""
    eval_users = list(val_pos.keys())
    hits, total = 0, 0
    with torch.no_grad():
        scores = score_fn(torch.tensor(eval_users))  # (len(eval_users), M)
        for row, u in enumerate(eval_users):
            s = scores[row].clone()
            for i in seen_by_user[u]:
                s[i] = -1e9  # прибираємо вже побачене
            topk = torch.topk(s, k).indices.tolist()
            truth = val_pos[u]
            hits += len(set(topk) & truth)
            total += min(len(truth), k)
    return hits / max(total, 1)

---
## Завдання 1. Vector Space Model (векторний підхід)

Перетворимо і книги, і користувачів на вектори в спільному просторі та шукатимемо рекомендації через cosine similarity. Роль ембединга книги відіграє її **нормалізований вектор жанрів** (пояснення про нормалізацію - нижче), а вектор користувача збираємо як **average pooling** ембедингів книг, які він уподобав.

**Що зробити:**

1. Побудуйте `item_emb` — матрицю L2-нормалізованих жанрових векторів усіх книг.
2. Реалізуйте функцію `user_vector(user_idx)` — зважене (за оцінкою) середнє ембедингів уподобаних книг користувача.
3. Реалізуйте функцію `vsm_scores(user_idxs)` — оцінки (cosine) усіх книг для набору користувачів, та порахуйте `recall_at_k`.
4. Покажіть топ-5 рекомендацій для одного користувача (з назвами книг).

**Довідка:**

L2-нормалізація — це ділення вектора на його довжину (L2-норму), щоб отримати вектор тієї ж напрямленості, але одиничної довжини.

Норма рахується як корінь із суми квадратів компонент:

$$\|v\|_2 = \sqrt{(v_1^2 + v_2^2 + \dots + v_n^2)}$$

а сам нормалізований вектор — це
$$\hat{v} = \frac{v}{\|v\|_2}$$

Навіщо це в рекомендаційних системах: після нормалізації **косинусна подібність зводиться до простого скалярного добутку**. Бо $\cos(a, b) = \frac{a \cdot b}{\|a\|\|b\|}$, і якщо обидва вектори вже одиничної довжини, знаменник = 1, тож $\cos(a,b) = a \cdot b$. Це і швидше, і прибирає вплив «довжини» вектора — порівнюється лише напрямок (тобто склад жанрів/смаків), а не те, скільки книг користувач оцінив.

*Приклад:*

Вектор `[3, 4]` має довжину $\sqrt{(9+16)}=5$, після нормалізації стає `[0.6, 0.8]` — напрямок той самий, довжина 1.

In [8]:
import torch.nn.functional as F

item_emb = F.normalize(item_feats, p=2, dim=1)

def user_vector(user_idx):
    user_id = users[user_idx]

    user_data = train_df[train_df["user_id"] == user_id]

    liked_data = user_data[user_data["rating"] >= LIKE_THRESHOLD]

    if len(liked_data) == 0:
        return torch.zeros(n_genres)

    item_idxs = [item_to_idx[b] for b in liked_data["book_id"]]

    ratings = torch.tensor(liked_data["rating"].values, dtype=torch.float32).unsqueeze(1)

    embs = item_emb[item_idxs]

    user_vec = (embs * ratings).sum(dim=0) / ratings.sum()

    user_vec = F.normalize(user_vec, p=2, dim=0)

    return user_vec

def vsm_scores(user_idxs):
    u_vecs = torch.stack([user_vector(u.item()) for u in user_idxs])

    scores = torch.matmul(u_vecs, item_emb.T)
    return scores

recall = recall_at_k(vsm_scores, k=10)
print(f"Recall@10: {recall:.4f}")

test_user_idx = list(val_pos.keys())[0]
test_user_id = users[test_user_idx]

scores = vsm_scores(torch.tensor([test_user_idx]))[0]

for i in seen_by_user[test_user_idx]:
    scores[i] = -1e9

top5_idx = torch.topk(scores, 5).indices.tolist()

print(f"\nТоп-5 рекомендацій для користувача {test_user_id}:")
for i, idx in enumerate(top5_idx):
    book_id = items[idx]
    print(f"{i+1}. {title_of[book_id]}")

Recall@10: 0.0522

Топ-5 рекомендацій для користувача 8765:
1. The Battle of the Labyrinth (Percy Jackson and the Olympians, #4)
2. Hush, Hush (Hush, Hush, #1)
3. Fallen (Fallen, #1)
4. Shadow Kiss (Vampire Academy, #3)
5. Beautiful Creatures (Caster Chronicles, #1)


**Питання:** Recall@10 у векторного підходу досить низький. Чому?


Ми описуємо книги лише через 12 бінарних жанрів. Через це десятки чи навіть сотні книг можуть мати абсолютно однаковий вектор.

---
## Завдання 2. Two-Tower архітектура

У Завданні 1 вектор користувача рахувався «вручну». Two-Tower натомість **навчає дві окремі башти**: User Tower (з ембединга user_id) та Item Tower (з жанрових ознак). Мережа зводить вектори уподобаних пар близько, а випадкових — далеко. Перевага: вектори книг рахуються один раз і кладуться в індекс (наприклад, FAISS) для швидкого retrieval — рахувати в реальному часі треба лише вектор користувача. Це **late fusion**.

**Що зробити:**

1. Реалізуйте `TwoTower` (user_tower через `nn.Embedding`, item_tower зі жанрових ознак), виходи L2-нормалізуйте.
2. Навчіть на лайках як позитивах і **negative sampling з усього корпусу** (як у пейпері від YouTube) з `BCEWithLogitsLoss` - він є реалізований в PyTorch.
3. Порахуйте `recall_at_k` через попередньо обчислені вектори книг і покажіть приклад рекомендацій.

> **Підказка.** Множте логіти на «температуру» (\~10), бо скалярний добуток нормалізованих векторів лежить у [-1, 1].
> Множення на температуру (\~10) розтягує діапазон логітів до [-10, 10], і тоді сигмоїда може видавати по-справжньому впевнені ймовірності (близькі до 0 і 1). Це дає лосу нормальний градієнт і модель навчається.


In [9]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class TwoTower(nn.Module):
    def __init__(self, n_users, n_genres, item_feats_tensor, emb_dim=32, temp=10.0):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)

        self.item_proj = nn.Linear(n_genres, emb_dim)

        self.register_buffer("item_feats", item_feats_tensor)
        self.temp = temp

    def forward(self, u_idx, i_idx):
        u_vec = self.user_emb(u_idx)
        i_feats_batch = self.item_feats[i_idx]
        i_vec = self.item_proj(i_feats_batch)

        u_vec = F.normalize(u_vec, p=2, dim=1)
        i_vec = F.normalize(i_vec, p=2, dim=1)

        scores = (u_vec * i_vec).sum(dim=1)

        return scores * self.temp

emb_dim = 32
model_tt = TwoTower(len(users), n_genres, item_feats, emb_dim=emb_dim)
optimizer = optim.Adam(model_tt.parameters(), lr=0.01)
criterion = nn.BCEWithLogitsLoss()

batch_size = 2048
dataset = TensorDataset(pos_u, pos_i)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

epochs = 10
model_tt.train()
for epoch in range(epochs):
    total_loss = 0
    for u_batch, i_batch in loader:
        neg_i_batch = torch.randint(0, M, size=(len(u_batch),))

        u_all = torch.cat([u_batch, u_batch])
        i_all = torch.cat([i_batch, neg_i_batch])

        labels = torch.cat([torch.ones(len(u_batch)), torch.zeros(len(neg_i_batch))])

        optimizer.zero_grad()
        logits = model_tt(u_all, i_all)

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1:02d} | Loss: {total_loss/len(loader):.4f}")

model_tt.eval()
with torch.no_grad():
    all_item_embs = F.normalize(model_tt.item_proj(item_feats), p=2, dim=1)

def tt_scores(user_idxs):
    u_vecs = F.normalize(model_tt.user_emb(user_idxs), p=2, dim=1)

    scores = torch.matmul(u_vecs, all_item_embs.T)
    return scores

recall_tt = recall_at_k(tt_scores, k=10)
print(f"\nTwo-Tower Recall@10: {recall_tt:.4f}")

scores = tt_scores(torch.tensor([test_user_idx]))[0]
for i in seen_by_user[test_user_idx]:
    scores[i] = -1e9
top5_idx = torch.topk(scores, 5).indices.tolist()

print(f"\nТоп-5 рекомендацій (Two-Tower) для користувача {test_user_id}:")
for i, idx in enumerate(top5_idx):
    book_id = items[idx]
    print(f"{i+1}. {title_of[book_id]}")

Epoch 01 | Loss: 0.8547
Epoch 02 | Loss: 0.6997
Epoch 03 | Loss: 0.6791
Epoch 04 | Loss: 0.6660
Epoch 05 | Loss: 0.6466
Epoch 06 | Loss: 0.6293
Epoch 07 | Loss: 0.6144
Epoch 08 | Loss: 0.6030
Epoch 09 | Loss: 0.5963
Epoch 10 | Loss: 0.5897

Two-Tower Recall@10: 0.0416

Топ-5 рекомендацій (Two-Tower) для користувача 8765:
1. Shadow of Night (All Souls Trilogy, #2)
2. Onyx (Lux, #2)
3. City of Bones (The Mortal Instruments, #1)
4. The Battle of the Labyrinth (Percy Jackson and the Olympians, #4)
5. Fallen (Fallen, #1)


---
## Завдання 3. Concat-based ranking (NCF)

На відміну від Two-Tower (late fusion), тут **early fusion**: склеюємо ембединг користувача і ознаки книги в один вектор і пропускаємо через MLP, який сам моделює крос-взаємодії. Платою є те, що модель **не можна заіндексувати** — щоб знайти найкращу книгу, треба прогнати кожну пару (user, item). Тому її використовують лише на фінальному ранжуванні кількох кандидатів.

**Що зробити:**

1. Реалізуйте `NCF`: `concat(user_embedding, item_genre_features)` → MLP → один логіт.
2. Навчіть на тих самих позитивах/негативах.
3. Реалізуйте `rank_ncf(user_idx, candidate_idxs)` — ранжування заданого списку кандидатів за `sigmoid` логіта.


In [10]:
class NCF(nn.Module):
    def __init__(self, n_users, n_genres, item_feats_tensor, emb_dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)

        self.register_buffer("item_feats", item_feats_tensor)

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim + n_genres, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1) # Один логіт на виході
        )

    def forward(self, u_idx, i_idx):
        u_vec = self.user_emb(u_idx)
        i_feats_batch = self.item_feats[i_idx]

        cat_vec = torch.cat([u_vec, i_feats_batch], dim=1)

        return self.mlp(cat_vec).squeeze(-1)

model_ncf = NCF(len(users), n_genres, item_feats, emb_dim=32)
optimizer_ncf = optim.Adam(model_ncf.parameters(), lr=0.01)
criterion_ncf = nn.BCEWithLogitsLoss()

model_ncf.train()
epochs = 10
for epoch in range(epochs):
    total_loss = 0
    for u_batch, i_batch in loader:
        neg_i_batch = torch.randint(0, M, size=(len(u_batch),))

        u_all = torch.cat([u_batch, u_batch])
        i_all = torch.cat([i_batch, neg_i_batch])
        labels = torch.cat([torch.ones(len(u_batch)), torch.zeros(len(neg_i_batch))])

        optimizer_ncf.zero_grad()
        logits = model_ncf(u_all, i_all)

        loss = criterion_ncf(logits, labels)
        loss.backward()
        optimizer_ncf.step()

        total_loss += loss.item()

    print(f"NCF Epoch {epoch+1:02d} | Loss: {total_loss/len(loader):.4f}")

def rank_ncf(user_idx, candidate_idxs):
    model_ncf.eval()
    with torch.no_grad():
        u_tensor = torch.tensor([user_idx] * len(candidate_idxs))
        i_tensor = torch.tensor(candidate_idxs)

        logits = model_ncf(u_tensor, i_tensor)
        probs = torch.sigmoid(logits)

    sorted_indices = torch.argsort(probs, descending=True).tolist()
    ranked_candidates = [candidate_idxs[i] for i in sorted_indices]

    return ranked_candidates

test_candidates = [i for i in range(M) if i not in seen_by_user[test_user_idx]]
ranked_ncf = rank_ncf(test_user_idx, test_candidates)

print(f"\nТоп-5 рекомендацій (NCF) для користувача {test_user_id}:")
for i in range(5):
    book_id = items[ranked_ncf[i]]
    print(f"{i+1}. {title_of[book_id]}")

NCF Epoch 01 | Loss: 0.6827
NCF Epoch 02 | Loss: 0.6606
NCF Epoch 03 | Loss: 0.6260
NCF Epoch 04 | Loss: 0.5990
NCF Epoch 05 | Loss: 0.5833
NCF Epoch 06 | Loss: 0.5725
NCF Epoch 07 | Loss: 0.5631
NCF Epoch 08 | Loss: 0.5572
NCF Epoch 09 | Loss: 0.5523
NCF Epoch 10 | Loss: 0.5461

Топ-5 рекомендацій (NCF) для користувача 8765:
1. The Host (The Host, #1)
2. The Hunger Games Trilogy Boxset (The Hunger Games, #1-3)
3. The Hunger Games (The Hunger Games, #1)
4. Catching Fire (The Hunger Games, #2)
5. Midnight Sun (Twilight, #1.5)


---
## Завдання 4. Двоетапний пайплайн Retrieval → Ranking

Поєднаємо все так, як це працює у великих системах: **Two-Tower швидко відбирає кандидатів** (retrieval серед усіх книг), а **NCF точно ранжує** цю коротку добірку.

**Що зробити:**

1. `retrieve(user_idx, n_candidates)` — топ-N книг за Two-Tower (Завдання 2), без уже побачених.
2. `recommend_pipeline(user_idx, n_candidates, top_k)` — прогнати кандидатів через `rank_ncf` (Завдання 3).
3. Показати для кількох користувачів: що відібрав retrieval і що залишив ranking.


In [11]:
def retrieve(user_idx, n_candidates=100):
    scores = tt_scores(torch.tensor([user_idx]))[0]

    for i in seen_by_user[user_idx]:
        scores[i] = -1e9

    top_n_idx = torch.topk(scores, n_candidates).indices.tolist()
    return top_n_idx

def recommend_pipeline(user_idx, n_candidates=100, top_k=5):
    candidates = retrieve(user_idx, n_candidates)

    ranked_candidates = rank_ncf(user_idx, candidates)

    return ranked_candidates[:top_k]

test_users_to_show = list(val_pos.keys())[:2]

for uid in test_users_to_show:
    print(f"Користувач ID: {users[uid]}")

    retrieved = retrieve(uid, n_candidates=100)
    print("Топ-5 швидкий відбір Two-Tower:")
    for i in range(5):
        print(f"  - {title_of[items[retrieved[i]]]}")

    final_top = recommend_pipeline(uid, n_candidates=100, top_k=5)
    print("\nТоп-5 точне сортування NCF:")
    for i, idx in enumerate(final_top):
        print(f"  {i+1}. {title_of[items[idx]]}")
    print("\n")

Користувач ID: 8765
Топ-5 швидкий відбір Two-Tower:
  - City of Bones (The Mortal Instruments, #1)
  - Onyx (Lux, #2)
  - Shadow of Night (All Souls Trilogy, #2)
  - Evermore (The Immortals, #1)
  - The Raven Boys (The Raven Cycle, #1)

Топ-5 точне сортування NCF:
  1. Catching Fire (The Hunger Games, #2)
  2. The Hunger Games (The Hunger Games, #1)
  3. The Host (The Host, #1)
  4. The Hunger Games Trilogy Boxset (The Hunger Games, #1-3)
  5. The Elite (The Selection, #2)


Користувач ID: 28776
Топ-5 швидкий відбір Two-Tower:
  - The Westing Game
  - Walk Two Moons
  - Harriet the Spy (Harriet the Spy #1)
  - From the Mixed-Up Files of Mrs. Basil E. Frankweiler
  - Hoot

Топ-5 точне сортування NCF:
  1. To Kill a Mockingbird
  2. Peace Like a River
  3. The Penultimate Peril (A Series of Unfortunate Events, #12)
  4. The Reptile Room (A Series of Unfortunate Events, #2)
  5. The Carnivorous Carnival (A Series of Unfortunate Events, #9)




**Питання:** навіщо ділити на два етапи, якщо можна ранжувати NCF одразу всі книги?

Через швидкість та масштабування

---
## Завдання 5. Теоретичний блок (письмові відповіді)

Спираючись на лекцію та на те, що Ви щойно побачили на реальних даних, дайте розгорнуті відповіді в markdown-клітинці нижче.

1. **Чому Recall@10 такий низький?** На реальних даних усі моделі цього ДЗ дають скромний Recall@10. Назвіть щонайменше дві причини (підказки: бідні контентні ознаки — лише 12 жанрів; розрідженість; те, що val-лайки не охоплюють усіх книг, які користувач *міг би* вподобати).
2. **Як покращити якість, не змінюючи архітектуру?** Які додаткові ознаки книг і користувачів з Goodbooks можна було б під'єднати? (автор, рік, середній рейтинг, повний набір тегів через TF-IDF, текстові ембединги опису через BERT...)
3. **Diversity.** Якщо користувач любить фентезі, чому не варто показувати йому 10 фентезі-книг підряд? Як технічно підмішати різноманітність?
4. **Freshness / cold start.** Нова книга має 0 оцінок. Який підхід цього ДЗ зможе рекомендувати її одразу, а який — ні? Чому?
5. **Watch time > CTR (з лекції).** Поясніть, чому YouTube оптимізує час перегляду, а не CTR, і як це технічно вшито у weighted logistic regression.


1.   Книги описали лише за допомогою 12 бінарних жанрів. Через це сотні абсолютно різних книг мають ідентичні вектори. Щільність матриці взаємодій складає менше 5%. Понад 95% можливих пар користувач-книга залишаються невідомими.
2.   Розширення ознак книг (авторів, рік видання, статистику рейтингів).
3.   Може набриднути одноманітність. Якщо модель трохи помилилася, то всі рекомендації будуть неактуальними. Користувачі самі не знають, що їм може сподобатися, поки не побачать це. Використати бізнес-правила. Змішати видачу з різних топів.
4.   Усі реалізовані підходи зможуть порекомендувати нову книгу одразу.
5.   Час перегляду є набагато чеснішим індикатором того, чи справді відео сподобалося людині. Більше часу на платформі означає більше можливостей показати рекламу. Інженери додали ваги. Для позитивних прикладів вага дорівнює фактичному часу перегляду відео.